In [25]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain
from collections import Counter

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [26]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [27]:
class ArticleCitations(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_citations(self):
        self.citations = self.db.sql("SELECT work_id AS citer_id, unnest(referenced_works) AS cited_id FROM econ.cited").df()
        print(f'{self.citations.shape = }\n{self.citations.head()}')
        return
    
    def make_citation_graph(self):
        vertices = set(self.citations.citer_id.tolist() + self.citations.cited_id.tolist())
        vs = pd.DataFrame(data=list(vertices), columns=['id'])
        vs['label'] = 'dummy'
        print(f'{vs.shape = }\n{vs.head()}')
        self.g = ig.Graph.DataFrame(self.citations, directed=True, use_vids=False, vertices=vs)
        summary = ig.summary(self.g, verbosity=1, width=256, edge_list_format='auto', max_rows=2, print_graph_attributes=True, 
                                          print_vertex_attributes=True, print_edge_attributes=True, full=False)
        print(f'*** SUMMARY OF self.g\n{summary}')
        return
    
    def run_pagerank(self):
        print('pageRank')
        damping = 0.85 if self.kind == 'both' else 0.85
        print(f'>> RUN pagerank with {damping = } for {self.kind = }')
        ranks = self.g.pagerank(damping=damping, weights='weights', implementation='prpack')
        print(f'>> CHECK pageRank  - should sum to unity {sum(ranks) = }')
        rank_max = max(ranks)
        ranks = [r/rank_max for r in ranks]
        self.pagerank = pd.DataFrame(zip(ranks, self.g.vs["name"], self.g.vs['label'], self.g.strength(mode='in', loops=False, weights='weights')),
                                      columns=['pageRank','citer', 'label', 'cite_count']).sort_values('pageRank', ascending=False)
        df = self.pagerank
        self.db.sql(f"CREATE OR REPLACE TABLE econ.pagerank_{self.kind} AS SELECT * FROM df")
        return
    
    def report_pagerank(self):
        kind = self.kind
        print(f'report pagerank {kind = }')
        df = self.db.sql(f"SELECT * FROM econ.pagerank_{kind}").df().sort_values('pageRank', ascending=False).reset_index(drop=True)        
        if self.kind == 'both':
            print(f'  Both df\n{df.head()}')
        elif self.kind == 'sources':
            print(f'  Source df\n{df[df.citer.str.contains('/S')].head()}')
        else:
            print(f'  Institutions df\n{df[df.citer.str.contains('/I')].head()}')           
        print(f'Sum of citations {df['cite_count'].sum() = }')
        print(f'Sum of scaled pageranks {df['pageRank'].sum() = }')
        self.db.sql("SELECT count(DISTINCT work_id) AS original_works_count FROM works").show()
        return        

In [28]:
def main():

    ac =ArticleCitations()
    ac.extract_citations()
    ac.make_citation_graph()
    
    return

In [ ]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬─────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────